In [ ]:
import pickle
import json
from pprint import pprint as pp
import io
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, BaseCrossValidator
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor



## Preprocessing Pipeline

In [ ]:
def make_preprocessor(numeric_cols=None, categorical_cols=None, scale_numeric=True):
    """
    Builds a universal preprocessing pipeline with optional scaling.
    Safety handles missing, empty, or completely absent numeric/categorical feature sets.
    """
    transformers = []

    # Handle Numeric Columns
    if numeric_cols is not None and len(numeric_cols) > 0:
        if scale_numeric:
            transformers.append(
                ('num', StandardScaler(), numeric_cols)
            )
        else:
            transformers.append(
                ('num', 'passthrough', numeric_cols)
            )
    # Handle Categorical Columns
    if categorical_cols is not None and len(categorical_cols) > 0:
        transformers.append(
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        )

    # Guard against completely empty configuration
    if not transformers:
        raise ValueError(
            "Universal preprocessor failed: You must provide at least one "
            "valid column in 'numerical_cols' or 'categorical_cols'."
        )

    # remainder='passthrough' keeps any unlisted columns untouched if needed
    return ColumnTransformer(transformers, remainder='drop')

def build_pipeline(preprocessor, model):
    """
    Creates a modeling pipeline combining preprocessing and model.
    """

    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    return pipe


## Basic Model Training Helper

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    All-in-one evaluator
    Auto-detects task type (Classification vs Regression) and reutnrs appropriate metrics
    """

    y_pred = model.predict(X_test)

    # --- Robust Task Aut0-Detection ---
    # Start by checking the top-level estimator
    estimator_type = getattr(model, "_estimator_type", None)

    # If it's a pipeline, inspect the final execution step
    if estimator_type is not None and hasattr(model, "steps"):
        final_step_model = model.steps[-1][1]
        estimator_type = getattr(final_step_model, "_estimator_type", None)


    # Method 1: Check if the model explicitely declares itself as a classifier
    is_classifier = (estimator_type == "classifier")


    # ---- CLASSIFIER WORKFLOW ----
    if is_classifier:
        # Get the actual model object instsance from the final pipeline step
        final_estimator = model.steps[-1][1] if hasattr(model, "steps") else model

        # If classifier supports predict_proba
        if hasattr(final_estimator, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
        elif hasattr(final_estimator, "decision_function"):
            y_proba = model.decision_function(X_test)
        else:
            # Fallbacl to hard predictions if no continuous score exists
            y_proba = y_pred

        # Extract classes safely (handles pipelines or raw estimators)
        labels = getattr(model, "classes_", None)
        if labels is None and hasattr(model, "steps"):
            labels = getattr(model.steps[-1][1], "classes_", None)

        metrics = {
            "task_type": "classification",
            "test_auc": roc_auc_score(y_test, y_proba),
            "confusion_matrix": confusion_matrix(y_test, y_pred, labels=labels),
            "report": classification_report(y_test, y_pred, output_dict=False, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
        }

        return metrics

    # ---- REGRESSION WORKFLOW ----
    else:
        # Calculate standard continuous metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Calculate Adjusted R² of X_test shape is accessible
        n = X_test.shape[0]
        p = X_test.shape[1] if hasattr(X_test, "shape") else 1
        adj_r2 = 1 - (1 - r2) * (n-1) / (n - p - 1) if n > (p + 1) else r2

        metrics = {
            "task_type": "regression",
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "adjusted_r2": adj_r2,
            "y_test": y_test,  # added for plotting
            "y_pred": y_pred   # added for plotting
        }

        return metrics

## GridSearchCV Wrapper

In [ ]:
def run_grid_search(pipe, param_grid, X_train, y_train, scoring, cv=5, groups=None):
    """
    Runs GridSearchCV on a given pipeline
    """
    grid = GridSearchCV(pipe, param_grid,
                        scoring=scoring,
                        cv=cv,
                        n_jobs=-1,
                        verbose=1
                       )

    # Base argument for fit
    fit_params = {}
    if isinstance(cv, BaseCrossValidator) and "Group" in type(cv).__name__:
        if groups is None:
            raise ValueError(
                f"The cross-validation splitter '{type(cv).__name__}' requires"
                "the 'groups' parameter, but 'groups=None' was provided."
            )
        fit_params["groups"] = groups

    grid.fit(X_train, y_train, **fit_params)

    return grid

## Multi-models Loop

In [ ]:
def run_multiple_models(models, grids, preprocessor,
                       X_train, y_train, X_test, y_test, scoring, **kwargs):
    """
    Runs grid search for multiple models and stores the results.
    """

    results = {}

    for label, model in models.items():
        print(f"\n===== Training {label} =====\n")

        pipe = build_pipeline(preprocessor, model)
        param_grid = grids[label]

        grid = run_grid_search(pipe, param_grid, X_train, y_train, scoring=scoring, **kwargs)

        # Best estimator
        best_model = grid.best_estimator_

        # Evaluate
        metrics = evaluate_model(best_model, X_test, y_test)

        results[label] = {
            "best_model": best_model,
            "best_prams": grid.best_params_,
            "cv_best_score": grid.best_score_,
            "test_metrics": metrics,
            "grid": grid
        }

        cm = metrics.get('confusion_matrix', None)

        display_model_results(label, grid, metrics, cm, best_model)


    return results

## Models Librairy

In [ ]:

all_models = {
    # ---- Linear Models ----
    "lin_reg": LinearRegression(),
    # "log_reg": LogisticRegression(max_iter=500),

    # ---- Decision Trees ----
    # "dt_reg": DecisionTreeRegressor(),
    # "dt_clf": DecisionTreeClassifier(),

    # ---- Random Forests ----
    "rf_reg": RandomForestRegressor(),
    # "rf_clf": RandomForestClassifier(),

    # ---- Boosting -----
    # "gb_clf": GradientBoostingClassifier(),
    # "ada_clf": AdaBoostClassifier(),
    # "xgb_clf": XGBClassifier(
    #     objective="binary:logistic",
    #     eval_metric="logloss",
    #     tree_method="hist"
    # ),
    "xgb_reg": XGBRegressor(
        objective="reg:squarederror",
    ),

    # ---- Bagging -----
    # "bag_clf": BaggingClassifier(estimator=DecisionTreeClassifier()),

    # ---- SVM -----
    "lin_svc": LinearSVC(),
    "svc_poly": SVC(kernel='poly', probability=True),
    "svc_rbf": SVC(kernel='rbf', probability=True),
    # "svr": SVR(kernel='rbf')

}

## Parameter Grids

In [ ]:
# Basic parameter grids
all_param_grids = {

    # ---- Linear Regression (no params) ----
    "lin_reg": {},

    # ---- Random Forest Regressor ----
    "rf_reg": {
        "model__n_estimators": [10, 100, 500],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 4],
        "model__min_samples_split": [2, 5, 10],
        "model__max_features": ['sqrt', 'log2']
    },

    # ---- XGBoost Regressor ----
    "xgb_reg": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    },

    # ---- LinearSVC ----
    "lin_svc": {
        "model__C": [0.1, 1, 10],
        "model__loss": ["hinge", "squared_hinge"]
    },

    # ---- SVC (Poly Kernel) ----
    "svc_poly": {
        "model__C": [0.1, 1, 10],
        "model__degree": [2,3,4],
        "model__gamma": ["scale", .01, .1],
        "model__coef0": [0, 1],
        "model__class_weight": [None, "balanced"]
    },

    # ---- SVC (RBF Kernel) ----
    "svc_rbf": {
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale",1, .1, .001, .0001],
        "model__class_weight": [None, "balanced"]
    },

    # ---- SVR ----
    "svr": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__epsilon": [0.1, 0.2, 0.5]
    }
}

## Visualization Helper Functions

In [ ]:
def show_shrunk(fig, width=480):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=300, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=buf.getvalue(), width=width))


def threshold_curve(y_test, y_proba):

    thresholds = np.linspace(0, 1, 200)
    precisions = []
    recalls = []
    f1s = []

    # High-DPI text rendering
    #plt.rcParams["figure.dpi"] = 200


    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
        recalls.append(recall_score(y_test, y_pred_t))
        f1s.append(f1_score(y_test, y_pred_t))

    fig, ax = plt.subplots(figsize=(8,5), dpi=150)

    ax.plot(thresholds, precisions, label="Precision")
    ax.plot(thresholds, recalls, label="Recall")
    ax.plot(thresholds, f1s, label="F1 Score")
    ax.axvline(0.5, color="grey", linestyle="--", label="Default t=0.50")

    box = ax.get_position()

    ax.set_position([
        box.x0 +0,
        box.y0 - 0,
        box.width * 1,
        box.height * 1
    ])

    ax.legend(loc="best", fontsize=8)
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.set_title("Precision–Recall–F1 vs Decision Threshold")
    #plt.tight_layout()
    #plt.show()
    show_shrunk(fig, width=600)




def display_model_results(name, grid, metrics, cm, best_model):
    """
    Display model results in a clean 2-column layout.
    Left: Text summary (Classification or Regression metrics)
    Right: Visual diagnostic (Confusion Matrix or Residuals Plot)
    """

    # High-DPI text rendering
    plt.rcParams["figure.dpi"] = 300

    fig = plt.figure(figsize=(26,4))
    gs = fig.add_gridspec(1, 2, wspace=0.15, width_ratios=[3,1])

    ax_text = fig.add_subplot(gs[0, 0])
    ax_text.axis("off")

    # --------------------------------------------------
    # BEST PARAMS (Grid or direct model params)
    # --------------------------------------------------
    if grid is None:
        # not from grid search → pull params from model
        try:
            raw_params = best_model.get_params()
            params_pretty = json.dumps(raw_params, indent=2)
        except:
            params_pretty = "N/A"
        cv_text = ""   # no CV score available
    else:
        params_pretty = json.dumps(grid.best_params_, indent=2)
        # Determine the appropriate label for the CV metric
        metric_name = "AUC (CV):  " if metrics["task_type"] == "classification" else "Score (CV):"
        cv_text = f"{metric_name}:   {grid.best_score_:.4f}\n"

    # --------------------------------------------------
    # Text block content
    # --------------------------------------------------
    # Classifier task
    if metrics["task_type"] == "classification":
        text = (
            f"============= {name} (Classification) =============\n\n"
            f"Best Params:\n{params_pretty}\n\n"
            f"{cv_text}"
            f"AUC (Test): {metrics['test_auc']:.4f}\n\n"
            f"F1 (Test):  {metrics['f1']:.4f}\n\n"
            f"Classification Report:\n{metrics['report']}"
        )
    # Regression task
    else:
        text = (
            f"============= {name} (Regression) =============\n\n"
            f"Best Params:\n{params_pretty}\n\n"
            f"{cv_text}"
            f"R² (Test):       {metrics['r2']:.4f}\n"
            f"Adjusted R²:     {metrics['adjusted_r2']:.4f}\n"
            f"RMSE (Test):     {metrics['rmse']:.4f}\n"
            f"MAE (Test):      {metrics['mae']:.4f}\n"
        )

    # Render text cleanly
    ax_text.text(
        0.0, 1.0, text,
        fontsize=9,
        va="top",
        ha="left",
        family="monospace",
        transform=ax_text.transAxes
    )


    # --------------------------------------------------
    # Right Column Visualizations
    # --------------------------------------------------
    ax_right = fig.add_subplot(gs[0, 1])

    if metrics["task_type"] == "classification":
        # Render Confusion Matrix
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=getattr(best_model, "classes_", ["0", "1"])
        )
        disp.plot(
            ax=ax_right,
            cmap="Blues",
            colorbar=False,
            values_format="d"
        )
        ax_right.set_title("Confusion Matrix", fontsize=10)

    else:
        # Render Regression Residuals / Prediction Fit Plot
        y_test_vals = metrics["y_test"]
        y_pred_vals = metrics["y_pred"]

        ax_right.scatter(y_test_vals, y_pred_vals, alpha=0.5, color="purple", edgecolors="w", s=20)

        # Draw a perfect diagonal identity reference line
        ideal_min = min(min(y_test_vals), min(y_pred_vals))
        ideal_max = max(max(y_test_vals), max(y_pred_vals))
        ax_right.plot([ideal_min, ideal_max], [ideal_min, ideal_max], 'r--', alpha=0.7, lw=1.5)

        ax_right.set_title("Actual vs. Predicted", fontsize=10)
        ax_right.set_xlabel("True Values", fontsize=8)
        ax_right.set_ylabel("Predictions", fontsize=8)
        ax_right.tick_params(labelsize=7)

    # Maintain your customized sub-axes positioning metrics
    box = ax_right.get_position()
    ax_right.set_position([
        box.x0 - .37,
        box.y0 + .45,
        box.width * 0.35,
        box.height * 0.35
    ])

    plt.show()


    # # --------------------------------------------------
    # # Confusion Matrix
    # # --------------------------------------------------
    #
    # ax_cm = fig.add_subplot(gs[0, 1])
    #
    # disp = ConfusionMatrixDisplay(
    #     confusion_matrix=cm,
    #     display_labels=getattr(best_model, "classes_", ["0", "1"])
    # )
    # disp.plot(
    #     ax=ax_cm,
    #     cmap="Blues",
    #     colorbar=False,
    #     values_format="d"
    # )
    #
    # # Make CM Smaller + Shift left
    # box = ax_cm.get_position()
    # ax_cm.set_position([
    #     box.x0 - .37,
    #     box.y0 + .45,
    #     box.width * 0.35,
    #     box.height * 0.35
    # ])
    #
    # ax_cm.set_title(f"Confusion Matrix", fontsize=10)
    #
    # #plt.tight_layout()
    # plt.show()


## Threshold tuning Helper Function

In [ ]:
def find_best_threshold(y_test, y_proba, beta=1):

    thresholds = np.linspace(0, 1, 200)

    # Initialize default values
    best_t = 0.5
    best_score = -1

    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        score = fbeta_score(y_test, y_pred_t, beta=beta, zero_division=0)

        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score




def eval_at_threshold(y_test, y_proba, threshold=0.5, beta=2):
    """
    Evaluate precision/recall/F1/Fβ at a custom decision threshold.

    Returns a dictionary of metrics:
        -threshold
        -precision
        -recall
        -f1
        -fbeta
        -confusion matrix
        -test_auc
        -classification report
    """

    # 1. Apply threshold → get binary predictions
    y_pred = (y_proba >= threshold).astype(int)

    # 2. Compute metrics
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    fbeta     = fbeta_score(y_test, y_pred, beta=beta, zero_division=0)


    # 3. Confusion matrix plot
    cm = confusion_matrix(y_test, y_pred)

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fbeta": fbeta,
        "confusion_matrix": cm,
        "test_auc": roc_auc_score(y_test, y_proba),
        "report": classification_report(y_test, y_pred, output_dict=False)
    }

---

# Load Dataset

In [ ]:
all_tests_v3_large_turbo = {}

# with open('./results/backup/export_all_tests_v3_large_turbo_500samples.pkl', 'rb') as f:
with open('./results/backup/rescued_all_tests_v3_large_turbo_ALL.pkl', 'rb') as f:
    all_tests_v3_large_turbo = pickle.load(f)

print(f"dataset keys: {all_tests_v3_large_turbo.keys()}")

In [ ]:
# Normalize each dataset into a list of dataframes
df_list = [pd.json_normalize(data) for data in all_tests_v3_large_turbo.values()]

# Combine them vertically and use the dictionary keys as a new columns to id them
df_tests = pd.concat(df_list, keys=all_tests_v3_large_turbo.keys()).reset_index(level=0, names="test_identifier")

In [ ]:
df_tests = pd.concat(df_list, keys=all_tests_v3_large_turbo.keys()).reset_index(level=0, names="test_identifier")
cols = [col.split(".") for col in df_tests.columns]

exclude = {"metrics", "asr_flags", "reasons"}

cleaned_cols = [
    ".".join(
        item.replace("c_analysis", "c_anly")
        .replace("n_analysis", "n_anly")
        .replace("digital_clipping_detected", "clipping_detected")
        for item in sublist
        if item not in exclude
    )
    for sublist in cols
]

test_id_map = {
    'whitenoise': 1,
    'synth-verb': 2,
    'real-verb': 3,
    'dropout': 4,
    'whitenoise+synth-verb+dropout': 5
}

df_tests["test_identifier"] = df_tests["test_identifier"].map(test_id_map)

df_tests.columns = cleaned_cols
df_tests.info()
# pprint(cleaned_cols)

In [ ]:
df_tests

In [ ]:
# == Some data engineering ==
df_tests["delta_wer"] = df_tests["n_wer"] - df_tests["c_wer"]
df_tests["delta_cer"] = df_tests["n_cer"] - df_tests["c_cer"]
df_tests["delta_crest_factor"] = df_tests["n_anly.crest_factor_db"] - df_tests["c_anly.crest_factor_db"]
df_tests["delta_mean_zcr"] = df_tests["n_anly.mean_zcr"] - df_tests["c_anly.mean_zcr"]
df_tests["delta_std_zcr"] = df_tests["n_anly.std_zcr"] - df_tests["c_anly.std_zcr"]
df_tests["delta_min_zcr"] = df_tests["n_anly.min_zcr"] - df_tests["c_anly.min_zcr"]
df_tests["delta_rms"] = df_tests["n_anly.rms_dbfs_global"] - df_tests["c_anly.rms_dbfs_global"]
df_tests.sort_values(by=["c_wer", "n_wer"], ascending=False, inplace=True)

df_tests[['c_wer', 'n_wer', 'delta_wer', 'c_cer', 'n_cer', 'delta_cer']].describe()

In [ ]:
# == generate column lists per type
num_cols = df_tests.select_dtypes(include=['number']).columns.tolist()
cat_cols = df_tests.select_dtypes(include=['str', 'object', 'category']).columns.tolist()

pp(f"numerical columns: {num_cols}")
pp(f"categorical columns: {cat_cols}")

In [ ]:
# == Look at cross-correlation between numerical variables ==
cols_to_correlate = [
    'delta_wer',
    # 'delta_cer',
    'n_anly.crest_factor_db',
    'n_anly.mean_zcr',
    'n_anly.rms_dbfs_global',
    'c_anly.mean_zcr',
    'c_anly.rms_dbfs_global',
    'delta_crest_factor',
    'delta_mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr',
    'delta_rms']

corr_df = df_tests[cols_to_correlate].dropna()
corr_matrix = corr_df.corr(method='spearman')

sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt='.2f',
            vmin=-1, vmax=1)
plt.title("Correlation: Audio Metrics vs WER Degradation")
plt.show()

In [ ]:
# ==  scatter plot of delta_wer and delta_zcr ==
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='n_anly.mean_zcr', y='delta_wer', data=df_tests, ax=axes[0], alpha=0.6)
axes[0].set_title('Noisy ZCR vs WER Degradation')
axes[0].set_xlabel("Noisy Audio ZCR")
axes[0].set_ylabel("WER delta")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='c_anly.rms_dbfs_global', y='delta_wer', data=df_tests, ax=axes[1], alpha=0.6)
axes[1].set_title('Original Clean RMS vs WER Degradation')
axes[1].set_xlabel('Clean RMS (dBFS)')
axes[1].set_ylabel('WER Delta')

plt.tight_layout()
plt.show()

In [ ]:
# Plot Noist ZCR vs Delta WER with colored dots for RMS dBFS
sns.scatterplot(
    x='n_anly.mean_zcr',
    y='delta_wer',
    hue='delta_min_zcr',
    palette='viridis',
    data=df_tests,
    alpha=0.8,
    s=60
)

plt.title('Noisy ZCR vs WER Degradation (Colored by delta_min_zcr)')
plt.xlabel('Noisy Audio ZCR')
plt.ylabel('WER Delta (Degradation)')

# Move the legend outside the plot
plt.legend(title='Delta min ZCR', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
df_tests

In [ ]:
df_tests[cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (whitenoise)
df_tests.loc[df_tests['test_identifier']==1, cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (synth-verb)
df_tests.loc[df_tests['test_identifier']==2, cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (real-verb)
df_tests.loc[df_tests['test_identifier']==3, cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (dropout)
df_tests.loc[df_tests['test_identifier']==4, cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (combined)
df_tests.loc[df_tests['test_identifier']==5, cols_to_correlate]

In [ ]:
# plot histograms of all metrics per tests
df_long = pd.melt(
    df_tests,
    id_vars=['test_identifier'],
    value_vars=cols_to_correlate,
    var_name="Metric_Name",
    value_name="Value"
)

sns.set_theme(style="ticks")
sns.set_context("talk")

grid = sns.FacetGrid(
    df_long,
    row='test_identifier',
    col="Metric_Name",
    sharex=False,  # Set to False if metrics have completely different numerical scales
    sharey=True,  # Set to False if some tests have vastly more samples than others
    margin_titles=True,
    height=4,
    aspect=1.3
)

# --- 4. Plot the Histograms ---
grid.map_dataframe(sns.histplot, x="Value", color="royalblue", bins=15)

# Adjust label padding so text doesn't overlap the plots
grid.figure.subplots_adjust(wspace=0.3, hspace=0.4)

# --- 5. Clean up layout ---
grid.tight_layout()
plt.savefig("./images/high_res_grid.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ==  Perform Train-Test Split on GroupsKFold ==
y = df_tests['delta_wer']
num_cols = ['n_anly.crest_factor_db',
             'n_anly.mean_zcr',
             'n_anly.rms_dbfs_global',
             'c_anly.crest_factor_db',
             'c_anly.mean_zcr',
             'c_anly.rms_dbfs_global',
             'delta_crest_factor',
             'delta_mean_zcr',
             'delta_std_zcr',
             'delta_min_zcr',
             'delta_rms']
X = df_tests[num_cols]

groups = df_tests["idx"].values
outer_gkf = GroupKFold(n_splits=5)

preprocessor = make_preprocessor(numeric_cols=num_cols, scale_numeric=True)
selected_models = ['lin_reg', 'rf_reg', 'xgb_reg']
models = {name: all_models[name] for name in selected_models}
param_grids = {name: all_param_grids[name] for name in selected_models}
nested_cv_results = {}

# Nested Cross Validation
for fold, (train_idx, test_idx) in enumerate(outer_gkf.split(X, y, groups=groups)):
    print(f"\n============ OUTER FOLD {fold+1} ============")

    # Generate strict train test split for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Slice the groups array to match only the training rows for this fold
    fold_groups = groups[train_idx]

    fold_results = run_multiple_models(
        models,
        param_grids,
        preprocessor,
        X_train,
        y_train,
        X_test,
        y_test,
        scoring="r2",
        cv=GroupKFold(n_splits=3),
        groups=fold_groups
    )

    nested_cv_results[f"fold_{fold+1}"] = fold_results



In [ ]:
nested_cv_results['fold_5']['xgb_reg']